# untold — training the gap-confidence model

Rule-based detectors + a small ML layer that scores each detected gap window.
This notebook rebuilds everything from scratch on Kaggle:

1. install `parshift` (the turn-taking library we use)
2. stage the project files (attach `untold_kaggle.zip` as an input dataset, or the project folder itself)
3. `python train.py` — synthetic gap injection, grouped 5-fold CV, saves the model
4. `python eval.py` — detector precision/recall/F1 against the hand-labelled ground truth

**What we got locally:** grouped CV `F1 = 0.823, P = 0.900, R = 0.800`, detector `F1 = 1.000`.
Kaggle's sklearn version may shift those a little.


In [ ]:
!pip install -q parshift

In [ ]:

import glob
import os
import shutil
import zipfile

work = "/kaggle/working/prob2"
os.makedirs(work, exist_ok=True)

for p in glob.glob("/kaggle/input/**/train.py", recursive=True):
    shutil.copytree(os.path.dirname(p), work, dirs_exist_ok=True)
    break
else:
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        zipfile.ZipFile(z).extractall(work)

hits = glob.glob(work + "/**/train.py", recursive=True)
assert hits, "train.py not found — attach untold_kaggle.zip (or the project folder) as a dataset"
os.chdir(os.path.dirname(hits[0]))
print(sorted(os.listdir()))


## Train

In [ ]:
!python train.py

In [ ]:
!python eval.py

In [ ]:

import json
from pathlib import Path

import matplotlib.pyplot as plt

m = json.loads(Path("artifacts/train_metrics.json").read_text())
print(m)

names = ["F1", "Precision", "Recall"]
vals = [m["cv_f1_mean"], m["cv_precision"], m["cv_recall"]]

plt.figure(figsize=(6, 3.5))
bars = plt.bar(names, vals, color=["#1976d2", "#43a047", "#fb8c00"])
plt.ylim(0, 1.1)
plt.title(f"grouped {m['cv_folds']}-fold CV  ({m['n_pos']} pos / {m['n_neg']} neg windows)")
for b, v in zip(bars, vals):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.03, f"{v:.3f}", ha="center", fontsize=11)
plt.tight_layout()
plt.show()


## The model on the demo conversations

In [ ]:

import json
from pathlib import Path

from gapdetect import analyze
from gapdetect.ml import confidence

for stem in ["team_standup", "design_review"]:
    case = json.loads(Path(f"data/{stem}.json").read_text(encoding="utf-8"))
    raw = "\n".join(f"{t['speaker']}: {t['text']}" for t in case["transcript"])
    rep = analyze(raw, title=case["title"])
    print(rep.title)
    for g in rep.gaps:
        c = confidence(g, rep.transcript)
        pct = f"{c:.0%}" if c is not None else "n/a"
        print(f"  {g.type.value:26} turns {g.turn_range}   conf {pct}")
